In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


In [1]:
import re
import string
import warnings
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from catboost import CatBoostClassifier
from catboost import CatBoostRegressor, Pool

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
from catboost import CatBoostRegressor, Pool
import holidays

In [2]:
PATH = "posts_vk.csv"

df = pd.read_csv(PATH, engine= "python")
df["text"] = df["text"].fillna("").astype(str)
df = df.query("text != ''")
print(df.shape)
display(df.head())
display(df.info())


(37425, 22)


,domain,owner_id,post_id,post_key,from_id,dt_msk,edited_dt_msk,text,views,likes,comments,reposts,reactions_total,is_pinned,marked_as_ads,is_donut,post_source_type,post_type,n_photos,cover_photo_url,all_photo_urls,post_url
0,literabook,-33874468,953135,-33874468_953135,-33874468,2026-04-02 11:33:58+03:00,NaN,"Интимнее, чем секс \nВсякий раз, когда мы гово...",31453,66,2,60,66.0,0,0,0,api,post,1,https://sun9-26.userapi.com/s/v1/ig2/N24sE2iTm...,"[""https://sun9-26.userapi.com/s/v1/ig2/N24sE2i...",https://vk.com/wall-33874468_953135
1,literabook,-33874468,953106,-33874468_953106,-33874468,2026-04-02 10:22:39+03:00,NaN,«В Япoнии тяжело быть жeной глaвы меcтной адми...,88292,612,9,301,612.0,0,0,0,api,post,1,https://sun9-65.userapi.com/s/v1/ig2/IWxPKVhWS...,"[""https://sun9-65.userapi.com/s/v1/ig2/IWxPKVh...",https://vk.com/wall-33874468_953106
2,literabook,-33874468,953054,-33874468_953054,-33874468,2026-04-02 08:46:20+03:00,NaN,Запомни три правила. ☝️☝️☝️\n \nПервое. Если т...,85588,323,4,151,323.0,0,0,0,api,post,2,https://sun1-94.userapi.com/s/v1/ig2/6tkrGWgDb...,"[""https://sun1-94.userapi.com/s/v1/ig2/6tkrGWg...",https://vk.com/wall-33874468_953054
3,literabook,-33874468,953015,-33874468_953015,-33874468,2026-04-02 07:16:36+03:00,NaN,"Итак, получается, у нас есть: 2 литра «Кока-Ко...",11774,217,2,132,217.0,0,0,0,api,post,2,https://sun9-24.userapi.com/s/v1/ig2/eWLc9yJPv...,"[""https://sun9-24.userapi.com/s/v1/ig2/eWLc9yJ...",https://vk.com/wall-33874468_953015
4,literabook,-33874468,952996,-33874468_952996,-33874468,2026-04-02 05:45:38+03:00,NaN,❗Стали известны заказчики беспрецендентного из...,269501,1839,291,1768,1839.0,0,0,0,api,post,5,https://sun1-84.userapi.com/s/v1/ig2/b0BmjLlT_...,"[""https://sun1-84.userapi.com/s/v1/ig2/b0BmjLl...",https://vk.com/wall-33874468_952996


<class 'pandas.core.frame.DataFrame'>
Index: 37425 entries, 0 to 39689
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   domain            37425 non-null  object 
 1   owner_id          37425 non-null  int64  
 2   post_id           37425 non-null  int64  
 3   post_key          37425 non-null  object 
 4   from_id           37425 non-null  int64  
 5   dt_msk            37425 non-null  object 
 6   edited_dt_msk     5043 non-null   object 
 7   text              37425 non-null  object 
 8   views             37425 non-null  int64  
 9   likes             37425 non-null  int64  
 10  comments          37425 non-null  int64  
 11  reposts           37425 non-null  int64  
 12  reactions_total   37388 non-null  float64
 13  is_pinned         37425 non-null  int64  
 14  marked_as_ads     37425 non-null  int64  
 15  is_donut          37425 non-null  int64  
 16  post_source_type  37425 non-null  object 
 17

None

In [3]:
def filter_out_iqr(group: pd.DataFrame) -> pd.DataFrame:
    q1 = group["views"].quantile(0.25)
    q3 = group["views"].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[(group["views"] >= lower) & (group["views"] <= upper)]


def clean_text_minimal(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.strip()

    return text


def base_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Unnamed: 0" in df.columns:
        df = df.drop(columns="Unnamed: 0")

    df["text"] = df["text"].apply(clean_text_minimal)
    df["dt_msk"] = pd.to_datetime(df["dt_msk"], errors="coerce")

    years = df["dt_msk"].dt.year.dropna().astype(int).unique().tolist()
    if years:
        ru_holidays = holidays.Russia(years=years)
        df["is_holiday"] = df["dt_msk"].dt.date.apply(
            lambda x: int(x in ru_holidays) if pd.notna(x) else 0
        )
    else:
        df["is_holiday"] = 0

    if {"domain", "views"}.issubset(df.columns):
        df = (
            df.groupby("domain", group_keys=False)
            .apply(filter_out_iqr)
            .reset_index(drop=True)
        )

    df = df.query("text != ''").copy()

    df["target_views"] = np.log1p(df["views"])

    df["engagement_rate"] = (
        (df["likes"].fillna(0) + df["comments"].fillna(0) + df["reposts"].fillna(0))
        / df["views"].replace(0, np.nan)
        * 100
    )

    drop_cols = [
        "post_type",
        "is_donut",
        "post_url",
        "marked_as_ads",
        "post_source_type",
        "views",
        "likes",
        "comments",
        "reposts",
        "reactions_total",
        "owner_id",
        "post_key",
        "from_id",
        "date_unix",
        "edited_dt_msk",
        "age_days",
        "cover_photo_url",
        "all_photo_urls",
        "is_pinned"
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df.reset_index(drop=True)
    return df

In [4]:
df = base_preprocessing(df)
df.head()

,domain,post_id,dt_msk,text,n_photos,is_holiday,target_views,engagement_rate
0,lentach,24937205,2026-04-02 11:19:31+03:00,Минцифры собирается сократить число мелких про...,1,0,12.101106,0.766924
1,lentach,24937094,2026-04-02 10:42:51+03:00,NASA впервые за 54 года отправила астронавтов ...,1,0,11.778991,0.831539
2,lentach,24936809,2026-04-01 21:48:33+03:00,Воздух и небо стали красными на острове Крит в...,4,0,11.508566,1.061629
3,lentach,24936731,2026-04-01 20:17:47+03:00,"В Италии женщина умерла после того, как её раз...",1,0,11.988855,0.933830
4,lentach,24936681,2026-04-01 18:44:36+03:00,Сегодня Apple исполнилось 50 лет.\nКомпания бы...,1,0,11.097592,0.701398


#Предобработка

In [5]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    dt = pd.to_datetime(df["dt_msk"], errors="coerce")

    df["hour"] = dt.dt.hour.fillna(0).astype("int16")
    df["weekday"] = dt.dt.weekday.fillna(0).astype("int16")
    df["month"] = dt.dt.month.fillna(0).astype("int16")
    df["day"] = dt.dt.day.fillna(0).astype("int16")

    df["is_weekend"] = (df["weekday"] >= 5).astype("int8")
    df["is_workday"] = (df["weekday"] < 5).astype("int8")
    df["is_friday"] = (df["weekday"] == 4).astype("int8")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

    df["n_photos"] = df["n_photos"].fillna(0).astype("int16")

    txt = df["text"].fillna("").astype(str)

    df["text_len_chars"] = txt.str.len().astype("int32")
    df["text_len_words"] = txt.str.split().str.len().fillna(0).astype("int32")
    df["num_lines"] = (txt.str.count("\n") + 1).astype("int16")

    df["num_exclam"] = txt.str.count("!").astype("int16")
    df["num_question"] = txt.str.count(r"\?").astype("int16")
    df["num_dots"] = txt.str.count(r"\.").astype("int16")
    df["num_commas"] = txt.str.count(",").astype("int16")
    df["num_colons"] = txt.str.count(":").astype("int16")

    df["num_hashtags"] = txt.str.count("#").astype("int16")
    df["num_links"] = (txt.str.count("http") + txt.str.count("vk.cc")).astype("int16")
    df["has_mention"] = txt.str.contains(r"\[club|\[id|@", regex=True, na=False).astype(
        "int8"
    )
    df["num_emojis"] = txt.str.count(r"[\U0001F300-\U0001FAFF]").astype("int16")

    letters = txt.str.findall(r"[A-Za-zА-Яа-яЁё]")
    caps = txt.str.findall(r"[A-ZА-ЯЁ]")
    df["caps_ratio"] = (
        (caps.str.len() / letters.str.len().replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )
    words = txt.str.findall(r"\w+", flags=re.UNICODE)
    digits = txt.str.findall(r"\d")
    caps_words = txt.str.findall(r"\b[A-ZА-ЯЁ]{2,}\b")

    df["unique_words_count"] = words.apply(
        lambda x: len(set(w.lower() for w in x))
    ).astype("int32")

    df["unique_words_ratio"] = (
        (df["unique_words_count"] / df["text_len_words"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["avg_word_len"] = words.apply(
        lambda x: np.mean([len(w) for w in x]) if len(x) else 0
    ).astype("float32")

    df["long_words_count"] = words.apply(lambda x: sum(len(w) >= 8 for w in x)).astype(
        "int16"
    )

    df["digit_count"] = digits.str.len().astype("int16")
    df["digit_ratio"] = (
        (df["digit_count"] / df["text_len_chars"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["uppercase_words_count"] = caps_words.str.len().astype("int16")

    sentence_count = txt.str.count(r"[.!?]+")
    df["sentence_count"] = np.where(
        df["text_len_chars"] > 0,
        sentence_count + 1,
        0,
    ).astype("int16")

    df["avg_sentence_len_words"] = (
        (df["text_len_words"] / df["sentence_count"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["ellipsis_count"] = txt.str.count(r"\.\.\.|…").astype("int16")
    df["repeat_punct_count"] = txt.str.count(r"[!?]{2,}").astype("int16")

    df["has_url"] = txt.str.contains(r"http|www|vk\.cc", regex=True, na=False).astype(
        "int8"
    )
    df["has_number"] = txt.str.contains(r"\d", regex=True, na=False).astype("int8")
    df["has_price"] = txt.str.contains(
        r"\d+\s?(₽|руб|р\b)|скидк|цена|бесплатно|акция",
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["starts_with_question"] = txt.str.match(
        r"^\s*[^a-zA-ZА-Яа-яЁё0-9]*.*\?", na=False
    ).astype("int8")
    df["starts_with_number"] = txt.str.match(r"^\s*\d", na=False).astype("int8")
    df["starts_with_emoji"] = txt.str.match(
        r"^\s*[\U0001F300-\U0001FAFF]", na=False
    ).astype("int8")

    cta_pattern = (
        r"пиши|смотри|читай|успей|переходи|жми|сохрани|подпишись|голосуй|оцени"
    )
    urgency_pattern = r"сегодня|сейчас|срочно|только сегодня|последний шанс|успей"

    df["has_call_to_action"] = txt.str.contains(
        cta_pattern,
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["has_urgency"] = txt.str.contains(
        urgency_pattern,
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["part_of_day"] = pd.cut(
        df["hour"],
        bins=[-1, 5, 11, 17, 23],
        labels=["night", "morning", "afternoon", "evening"],
    )

    df["text_len_bin"] = pd.cut(
        df["text_len_words"],
        bins=[-1, 5, 15, 30, 60, 10**9],
        labels=["very_short", "short", "medium", "long", "very_long"],
    )

    df["hour_weekday"] = df["hour"].astype(str) + "_" + df["weekday"].astype(str)
    df["domain_weekday"] = (
        df["domain"].fillna("unknown").astype(str) + "_" + df["weekday"].astype(str)
    )
    df["domain_part_of_day"] = (
        df["domain"].fillna("unknown").astype(str) + "_" + df["part_of_day"].astype(str)
    )
    df["weekend_hour"] = df["is_weekend"].astype(str) + "_" + df["hour"].astype(str)

    return df

In [6]:
data = make_features(df)
data.head()

,domain,post_id,dt_msk,text,n_photos,is_holiday,target_views,engagement_rate,hour,weekday,month,day,is_weekend,is_workday,is_friday,hour_sin,hour_cos,weekday_sin,weekday_cos,text_len_chars,text_len_words,num_lines,num_exclam,num_question,num_dots,num_commas,num_colons,num_hashtags,num_links,has_mention,num_emojis,caps_ratio,unique_words_count,unique_words_ratio,avg_word_len,long_words_count,digit_count,digit_ratio,uppercase_words_count,sentence_count,avg_sentence_len_words,ellipsis_count,repeat_punct_count,has_url,has_number,has_price,starts_with_question,starts_with_number,starts_with_emoji,has_call_to_action,has_urgency,part_of_day,text_len_bin,hour_weekday,domain_weekday,domain_part_of_day,weekend_hour
0,lentach,24937205,2026-04-02 11:19:31+03:00,Минцифры собирается сократить число мелких про...,1,0,12.101106,0.766924,11,3,4,2,0,1,0,0.258819,-9.659258e-01,0.433884,-0.900969,505,71,3,0,0,3,5,0,0,0,0,0,0.019139,65,0.915493,5.957747,26,5,0.009901,1,4,17.750000,0,0,0,1,0,0,0,0,0,0,morning,very_long,11_3,lentach_3,lentach_morning,0_11
1,lentach,24937094,2026-04-02 10:42:51+03:00,NASA впервые за 54 года отправила астронавтов ...,1,0,11.778991,0.831539,10,3,4,2,0,1,0,0.500000,-8.660254e-01,0.433884,-0.900969,519,80,3,0,0,5,2,0,0,0,0,0,0.060680,71,0.887500,5.108434,19,12,0.023121,3,6,13.333333,0,0,0,1,0,0,0,0,0,0,morning,very_long,10_3,lentach_3,lentach_morning,0_10
2,lentach,24936809,2026-04-01 21:48:33+03:00,Воздух и небо стали красными на острове Крит в...,4,0,11.508566,1.061629,21,2,4,1,0,1,0,-0.707107,7.071068e-01,0.974928,-0.222521,86,16,1,0,0,0,0,0,0,0,0,0,0.043478,16,1.000000,4.312500,2,0,0.000000,0,1,16.000000,0,0,0,0,0,0,0,0,0,0,evening,medium,21_2,lentach_2,lentach_evening,0_21
3,lentach,24936731,2026-04-01 20:17:47+03:00,"В Италии женщина умерла после того, как её раз...",1,0,11.988855,0.933830,20,2,4,1,0,1,0,-0.866025,5.000000e-01,0.974928,-0.222521,267,38,2,0,0,1,4,0,0,0,0,0,0.013514,37,0.973684,5.743590,12,2,0.007491,0,2,19.000000,0,0,0,1,0,0,0,0,0,0,evening,long,20_2,lentach_2,lentach_evening,0_20
4,lentach,24936681,2026-04-01 18:44:36+03:00,Сегодня Apple исполнилось 50 лет.\nКомпания бы...,1,0,11.097592,0.701398,18,2,4,1,0,1,0,-1.000000,-1.836970e-16,0.974928,-0.222521,126,19,2,0,0,1,1,0,0,0,0,0,0.090909,18,0.947368,5.578948,5,7,0.055556,0,2,9.500000,0,0,0,1,0,0,0,0,0,1,evening,medium,18_2,lentach_2,lentach_evening,0_18


###Метрики engagement_rate

In [7]:
def print_metrics_er(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"{name}:")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")
    print()

###Эксперименты с engagament rate

In [8]:
num_cols = ['n_photos',
       'hour', 'weekday', 'month', 'day', 'is_weekend',
       'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'text_len_chars',
       'text_len_words', 'num_lines', 'num_exclam', 'num_question', 'num_dots',
       'num_commas', 'num_colons', 'num_hashtags', 'num_links', 'has_mention',
       'num_emojis', 'caps_ratio', 'unique_words_count',
       'unique_words_ratio', 'avg_word_len', 'long_words_count', 'digit_count',
       'digit_ratio', 'uppercase_words_count', 'sentence_count',
       'avg_sentence_len_words', 'ellipsis_count', 'repeat_punct_count',
       'has_url', 'has_number', 'has_price', 'starts_with_question',
       'starts_with_number', 'starts_with_emoji', 'has_call_to_action',
       'has_urgency', 'is_holiday','is_workday','is_friday']

text_col = "text"
target_col = "engagement_rate"
cat_cols = [
    "domain",
    "part_of_day",
    "text_len_bin",
    "hour_weekday",
    "domain_weekday",
    "domain_part_of_day",
    "weekend_hour"
]

In [9]:
X = data[[text_col] + num_cols + cat_cols].copy()
y = data[target_col].values

# 70% train, 15% valid, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=50000,
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.95,
                sublinear_tf=True
            ),
            text_col
        ),
        (
            "num",
            StandardScaler(),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        )
    ],
    remainder="drop"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_valid = model.predict(X_valid)
y_pred_test = model.predict(X_test)

print_metrics_er(y_train, y_pred_train, "Train")
print_metrics_er(y_valid, y_pred_valid, "Validation")
print_metrics_er(y_test,y_pred_test, "Test")

Train:
  RMSE = 0.3164
  MAE  = 0.2181
  R2   = 0.7864

Validation:
  RMSE = 0.4621
  MAE  = 0.3296
  R2   = 0.5476

Test:
  RMSE = 0.4671
  MAE  = 0.3304
  R2   = 0.5393



In [10]:
text_col = ["text"]

cat_cols = [
    "domain", "part_of_day", "text_len_bin", "hour_weekday",
    "domain_weekday", "domain_part_of_day", "weekend_hour"
]
num_cols = [
    "n_photos", "hour", "weekday", "month", "day", "is_weekend",
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "text_len_chars",
    "text_len_words", "num_lines", "num_exclam", "num_question", "num_dots",
    "num_commas", "num_colons", "num_hashtags", "num_links", "has_mention",
    "num_emojis", "caps_ratio", "unique_words_count",
    "unique_words_ratio", "avg_word_len", "long_words_count", "digit_count",
    "digit_ratio", "uppercase_words_count", "sentence_count",
    "avg_sentence_len_words", "ellipsis_count", "repeat_punct_count",
    "has_url", "has_number", "has_price", "starts_with_question",
    "starts_with_number", "starts_with_emoji", "has_call_to_action",
    "has_urgency", "is_holiday", "is_workday", "is_friday"
]

feature_cols = text_col + num_cols + cat_cols

In [11]:
X_train_cb = X_train[feature_cols].copy()
X_valid_cb = X_valid[feature_cols].copy()
X_test_cb = X_test[feature_cols].copy()

train_pool = Pool(
    data=X_train_cb,
    label=y_train,
    text_features=text_col,
    cat_features=cat_cols
)

valid_pool = Pool(
    data=X_valid_cb,
    label=y_valid,
    text_features=text_col,
    cat_features=cat_cols
)

test_pool = Pool(
    data=X_test_cb,
    label=y_test,
    text_features=text_col,
    cat_features=cat_cols
)

model_cb = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=300
)

model_cb.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

y_pred_train_cb = model_cb.predict(train_pool)
y_pred_valid_cb = model_cb.predict(valid_pool)
y_pred_test_cb = model_cb.predict(test_pool)

print_metrics_er(y_train, y_pred_train_cb, "CB Train")
print_metrics_er(y_valid, y_pred_valid_cb, "CB Validation")
print_metrics_er(y_test, y_pred_test_cb, "CB Test")

0:	learn: 0.6744312	test: 0.6767858	best: 0.6767858 (0)	total: 113ms	remaining: 9m 22s
300:	learn: 0.4478755	test: 0.4540259	best: 0.4540259 (300)	total: 14.8s	remaining: 3m 50s
600:	learn: 0.4306360	test: 0.4458234	best: 0.4458234 (600)	total: 29s	remaining: 3m 32s
900:	learn: 0.4194912	test: 0.4420328	best: 0.4420328 (900)	total: 43s	remaining: 3m 15s
1200:	learn: 0.4100940	test: 0.4393534	best: 0.4393534 (1200)	total: 57.1s	remaining: 3m
1500:	learn: 0.4020130	test: 0.4374840	best: 0.4374840 (1500)	total: 1m 11s	remaining: 2m 46s
1800:	learn: 0.3945316	test: 0.4359379	best: 0.4359361 (1799)	total: 1m 25s	remaining: 2m 32s
2100:	learn: 0.3880271	test: 0.4349755	best: 0.4349669 (2099)	total: 1m 39s	remaining: 2m 17s
2400:	learn: 0.3818576	test: 0.4340126	best: 0.4340126 (2400)	total: 1m 54s	remaining: 2m 3s
2700:	learn: 0.3759913	test: 0.4334082	best: 0.4334004 (2680)	total: 2m 10s	remaining: 1m 50s
3000:	learn: 0.3703095	test: 0.4326430	best: 0.4326380 (2999)	total: 2m 24s	remaining:

In [12]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

MODEL_NAME = "cointegrated/rubert-tiny2"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

BATCH_SIZE = 64
MAX_LENGTH = 512

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_text = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model_text.eval()

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(83828, 312, padding_idx=0)
    (position_embeddings): Embedding(2048, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
   

In [14]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode_texts(texts, tokenizer, model, batch_size=64, max_length=128, device="cpu"):
    all_embeddings = []

    texts = pd.Series(texts).fillna("").astype(str).tolist()

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)

        emb = mean_pooling(outputs, encoded["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)

In [15]:
X_train_cb = X_train[feature_cols].copy()
X_valid_cb = X_valid[feature_cols].copy()
X_test_cb = X_test[feature_cols].copy()

train_emb = encode_texts(
    X_train_cb["text"],
    tokenizer=tokenizer,
    model=model_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE
)

valid_emb = encode_texts(
    X_valid_cb["text"],
    tokenizer=tokenizer,
    model=model_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE
)

test_emb = encode_texts(
    X_test_cb["text"],
    tokenizer=tokenizer,
    model=model_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE
)

print(train_emb.shape, valid_emb.shape, test_emb.shape)

  0%|          | 0/376 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

  0%|          | 0/81 [00:00<?, ?it/s]

(24005, 312) (5144, 312) (5144, 312)


In [16]:
emb_cols = [f"rubert_emb_{i}" for i in range(train_emb.shape[1])]

train_emb_df = pd.DataFrame(train_emb, index=X_train_cb.index, columns=emb_cols)
valid_emb_df = pd.DataFrame(valid_emb, index=X_valid_cb.index, columns=emb_cols)
test_emb_df  = pd.DataFrame(test_emb,  index=X_test_cb.index,  columns=emb_cols)
X_train_cb_emb = pd.concat(
    [X_train_cb.drop(columns=["text"]).reset_index(drop=True),
     train_emb_df.reset_index(drop=True)],
    axis=1
)

X_valid_cb_emb = pd.concat(
    [X_valid_cb.drop(columns=["text"]).reset_index(drop=True),
     valid_emb_df.reset_index(drop=True)],
    axis=1
)

X_test_cb_emb = pd.concat(
    [X_test_cb.drop(columns=["text"]).reset_index(drop=True),
     test_emb_df.reset_index(drop=True)],
    axis=1
)

train_pool_emb = Pool(
    data=X_train_cb_emb,
    label=y_train,
    cat_features=cat_cols
)

valid_pool_emb = Pool(
    data=X_valid_cb_emb,
    label=y_valid,
    cat_features=cat_cols
)

test_pool_emb = Pool(
    data=X_test_cb_emb,
    label=y_test,
    cat_features=cat_cols
)

In [17]:
model_cb_emb = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=300
)

model_cb_emb.fit(
    train_pool_emb,
    eval_set=valid_pool_emb,
    use_best_model=True
)

0:	learn: 0.6742912	test: 0.6767625	best: 0.6767625 (0)	total: 42.9ms	remaining: 3m 34s
300:	learn: 0.4330447	test: 0.4498778	best: 0.4498737 (299)	total: 6.44s	remaining: 1m 40s
600:	learn: 0.4036872	test: 0.4418643	best: 0.4418643 (600)	total: 12.9s	remaining: 1m 34s
900:	learn: 0.3806194	test: 0.4391313	best: 0.4391313 (900)	total: 19.3s	remaining: 1m 27s
1200:	learn: 0.3614545	test: 0.4372181	best: 0.4371963 (1198)	total: 25.6s	remaining: 1m 21s
1500:	learn: 0.3447400	test: 0.4357593	best: 0.4357593 (1500)	total: 32.4s	remaining: 1m 15s
1800:	learn: 0.3297272	test: 0.4342022	best: 0.4341811 (1798)	total: 39.4s	remaining: 1m 9s
2100:	learn: 0.3161154	test: 0.4333195	best: 0.4332821 (2062)	total: 45.7s	remaining: 1m 3s
2400:	learn: 0.3037473	test: 0.4325730	best: 0.4325730 (2400)	total: 51.9s	remaining: 56.2s
2700:	learn: 0.2921771	test: 0.4319391	best: 0.4319132 (2689)	total: 58.2s	remaining: 49.6s
3000:	learn: 0.2812083	test: 0.4313555	best: 0.4313409 (2986)	total: 1m 4s	remaining:

CatBoostRegressor(depth=6, early_stopping_rounds=200, eval_metric='RMSE', iterations=5000, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=300)

In [18]:
y_pred_train_cb_emb = model_cb_emb.predict(train_pool_emb)
y_pred_valid_cb_emb = model_cb_emb.predict(valid_pool_emb)
y_pred_test_cb_emb = model_cb_emb.predict(test_pool_emb)

In [19]:
print_metrics_er(y_train, y_pred_train_cb_emb, "CB Train")
print_metrics_er(y_valid, y_pred_valid_cb_emb, "CB Validation")
print_metrics_er(y_test, y_pred_test_cb_emb, "CB Test")

CB Train:
  RMSE = 0.2279
  MAE  = 0.1667
  R2   = 0.8892

CB Validation:
  RMSE = 0.4291
  MAE  = 0.3013
  R2   = 0.6100

CB Test:
  RMSE = 0.4299
  MAE  = 0.3027
  R2   = 0.6096



In [22]:
pip install lightgbm


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Добавление эмбеддингов CLIP

In [14]:
emb = pd.read_csv('clip_image_embeddings.csv')

In [16]:
emb_cols = [col for col in emb.columns if col.startswith("clip_img_emb_")]

emb["image_embedding"] = emb[emb_cols].values.tolist()

In [19]:
data = data.merge(
    emb[["post_id", "image_embedding"]],
    on="post_id",
    how="left"
)

In [24]:
emb_text = encode_texts(
    data["text"],
    tokenizer=tokenizer,
    model=model_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE
)

  0%|          | 0/536 [00:00<?, ?it/s]

In [27]:
data["text_embedding"] = emb_text.tolist()

In [36]:
def is_valid_embedding(x):
    return isinstance(x, list) and len(x) > 0

mask = (
    data["image_embedding"].apply(is_valid_embedding) &
    data["text_embedding"].apply(is_valid_embedding)
)

data_clean = data.loc[mask].copy()

print("До очистки:", data.shape)
print("После очистки:", data_clean.shape)

До очистки: (34293, 59)
После очистки: (34264, 59)


In [37]:
num_cols = ['n_photos',
       'hour', 'weekday', 'month', 'day', 'is_weekend',
       'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'text_len_chars',
       'text_len_words', 'num_lines', 'num_exclam', 'num_question', 'num_dots',
       'num_commas', 'num_colons', 'num_hashtags', 'num_links', 'has_mention',
       'num_emojis', 'caps_ratio', 'unique_words_count',
       'unique_words_ratio', 'avg_word_len', 'long_words_count', 'digit_count',
       'digit_ratio', 'uppercase_words_count', 'sentence_count',
       'avg_sentence_len_words', 'ellipsis_count', 'repeat_punct_count',
       'has_url', 'has_number', 'has_price', 'starts_with_question',
       'starts_with_number', 'starts_with_emoji', 'has_call_to_action',
       'has_urgency', 'is_holiday', 'is_workday', 'is_friday']

target_col = "engagement_rate"

cat_cols = [
    "domain",
    "part_of_day",
    "text_len_bin",
    "hour_weekday",
    "domain_weekday",
    "domain_part_of_day",
    "weekend_hour"
]


In [40]:
emb_cols = ["image_embedding", "text_embedding"]

X = data_clean[num_cols + cat_cols + emb_cols].copy()
y = data_clean[target_col].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

In [ ]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_cols,
    embedding_features=emb_cols
)

valid_pool = Pool(
    data=X_valid,
    label=y_valid,
    cat_features=cat_cols,
    embedding_features=emb_cols
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_cols,
    embedding_features=emb_cols
)

model_cb = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.01,
    depth=8,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=300
)

model_cb.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

y_pred_train_cb = model_cb.predict(train_pool)
y_pred_valid_cb = model_cb.predict(valid_pool)
y_pred_test_cb = model_cb.predict(test_pool)

print_metrics_er(y_train, y_pred_train_cb, "CB Train")
print_metrics_er(y_valid, y_pred_valid_cb, "CB Validation")
print_metrics_er(y_test, y_pred_test_cb, "CB Test")

0:	learn: 0.6844436	test: 0.6786201	best: 0.6786201 (0)	total: 9.24ms	remaining: 46.2s
300:	learn: 0.4651604	test: 0.4638440	best: 0.4638440 (300)	total: 2.94s	remaining: 45.8s
600:	learn: 0.4487774	test: 0.4539533	best: 0.4539533 (600)	total: 5.77s	remaining: 42.2s
900:	learn: 0.4389661	test: 0.4500467	best: 0.4500467 (900)	total: 8.62s	remaining: 39.2s
1200:	learn: 0.4296216	test: 0.4475549	best: 0.4475549 (1200)	total: 11.4s	remaining: 36.2s
1500:	learn: 0.4228141	test: 0.4461272	best: 0.4461247 (1499)	total: 14.2s	remaining: 33.2s
1800:	learn: 0.4173165	test: 0.4455230	best: 0.4455230 (1800)	total: 17s	remaining: 30.1s
2100:	learn: 0.4122178	test: 0.4450110	best: 0.4450095 (2099)	total: 19.7s	remaining: 27.2s
2400:	learn: 0.4074667	test: 0.4446846	best: 0.4446794 (2390)	total: 22.5s	remaining: 24.4s
2700:	learn: 0.4031947	test: 0.4444723	best: 0.4444573 (2689)	total: 25.4s	remaining: 21.6s
3000:	learn: 0.3988127	test: 0.4441675	best: 0.4441385 (2978)	total: 28.3s	remaining: 18.8s
3